# 📊 Analyse Olist — KPIs Business

Star Schema : 112k ventes, 99k clients, 33k produits, 634 jours.

In [ ]:
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 120

conn = sqlite3.connect('../data/olist_warehouse.db')

df = pd.read_sql_query("""
    SELECT f.*, d.year, d.month, d.day_of_week, d.is_weekend,
           c.state as customer_state,
           p.category_name_en as category
    FROM fact_sales f
    JOIN dim_time d ON f.time_key = d.time_key
    JOIN dim_customer c ON f.customer_key = c.customer_key
    JOIN dim_product p ON f.product_key = p.product_key
""", conn)

conn.close()
print(f'{len(df):,} lignes chargees')
df.head(3)

## 1. Chiffre d'affaires mensuel

In [ ]:
df['year_month'] = df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2)
monthly_revenue = df.groupby('year_month')['total_payment'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_revenue.index, monthly_revenue.values, marker='o', linewidth=1.5, markersize=3)
ax.set_title('Chiffre d\'affaires mensuel')
ax.set_xlabel('Mois')
ax.set_ylabel('CA (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.tick_params(axis='x', rotation=45)
every_nth = max(1, len(monthly_revenue) // 12)
for i, label in enumerate(ax.get_xticklabels()):
    if i % every_nth != 0:
        label.set_visible(False)
plt.tight_layout()
plt.savefig('../data/kpi_ca_mensuel.png')
plt.show()

print(f'CA total : R$ {monthly_revenue.sum():,.0f}')
print(f'Meilleur mois : {monthly_revenue.idxmax()} — R$ {monthly_revenue.max():,.0f}')

## 2. Top 10 categories

In [ ]:
top_cat = df.groupby('category')['total_payment'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_cat.index[::-1], top_cat.values[::-1], color='steelblue')
for bar, val in zip(bars, top_cat.values[::-1]):
    ax.text(bar.get_width() + 2000, bar.get_y() + bar.get_height()/2, f'R$ {val:,.0f}', va='center', fontsize=9)
ax.set_title('Top 10 categories par CA')
ax.set_xlabel('CA (R$)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('../data/kpi_top_categories.png')
plt.show()

## 3. Delai de livraison par etat

In [ ]:
delivery_by_state = df.groupby('customer_state')['delivery_days'].mean().sort_values().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(delivery_by_state.index, delivery_by_state.values, color='coral')
ax.axhline(df['delivery_days'].mean(), color='red', linestyle='--', label=f'Moyenne nationale : {df["delivery_days"].mean():.0f}j')
ax.set_title('Delai de livraison moyen par etat (Top 15 les plus rapides)')
ax.set_ylabel('Jours')
ax.legend()
plt.tight_layout()
plt.savefig('../data/kpi_delivery_state.png')
plt.show()

print(f'Delai moyen national : {df["delivery_days"].mean():.0f} jours')
print(f'% de commandes en retard : {df["is_delayed"].mean()*100:.1f}%')

## 4. Repartition des notes

In [ ]:
scores = df['avg_review_score'].value_counts().sort_index()
colors = ['gray', 'tomato', 'orange', 'gold', 'limegreen']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([f'{int(k)}★' for k in scores.index], scores.values, color=colors)
for bar, val in zip(bars, scores.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500, f'{val:,}', ha='center')
ax.set_title('Repartition des notes clients')
ax.set_ylabel('Nombre de commandes')
plt.tight_layout()
plt.savefig('../data/kpi_reviews.png')
plt.show()

print(f'Note moyenne : {df["avg_review_score"].mean():.2f} / 5')

## 5. CA par jour de la semaine

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_revenue = df.groupby('day_of_week')['total_payment'].sum().reindex(dow_order)
dow_weekend = ['skyblue' if d not in ['Saturday', 'Sunday'] else 'lightcoral' for d in dow_order]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(dow_revenue.index, dow_revenue.values, color=dow_weekend)
ax.set_title('CA par jour de la semaine')
ax.set_ylabel('CA (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../data/kpi_ca_dow.png')
plt.show()

## 6. Prix moyen par categorie (Top 10)

In [ ]:
cat_price = df.groupby('category')['unit_price'].mean().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(cat_price.index[::-1], cat_price.values[::-1], color='mediumseagreen')
for bar, val in zip(ax.containers[0], cat_price.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, f'R$ {val:.0f}', va='center')
ax.set_title('Prix unitaire moyen par categorie (Top 10)')
ax.set_xlabel('Prix moyen (R$)')
plt.tight_layout()
plt.savefig('../data/kpi_price_category.png')
plt.show()